Si affronta un problema tipico dell'Information Retrieval: usando VSM (ma anche gli altri modelli probabilistici BM25 e LM) non **siamo in grado di catturare la semantica dei documenti, ci limitiamo esclusivamente ad usare approcci del tipo term-match**. 

Questo comporta che, anche se due documenti parlano dello stesso argomento ma usano termini diversi, allora il modello li considererà come documenti poco simili tra loro. Un altro problema in questo senso è legato alle parole **polisemiche**: se una parola ha più significati, il modello classico non è in grado di distinguere bene il contesto.

**Latent Semantic Indexing (LSI)** è stato uno dei primi approcci per cercare di superare questi limiti.

### Richiami di Algebra Lineare
Dal momento che LSI non nasce come tecnica linguistica, ma come tecnica matriciale, è doveroso anzitutto fare un richiamo di algebra lineare per capirla. 

In IR, usando VSM, abbiamo una **matrice termini-documenti** $A$ (dove le righe sono i termini, le colonne i documenti, e i valori indicando il peso tf-idf del termine nel documento). LSI si basa su una tecnica di decomposizione di questa matrice, chiamata **Singular Value Decomposition (SVD)**.

#### Autovettori e autovalori

Gli **autovettori** e gli **autovalori** (eigenvectors e eigenvalues) rappresentano delle **proprietà proprie di una matrice**. In particolare, data $S$ matrice quadrata $n \times n$, si ha:
$$S \cdot v = \lambda \cdot v$$
dove $v$ è un autovettore e $\lambda$ è l'autovalore associato. 

Il significato è che se si moltiplica la matrice $S$ per il suo autovettore $v$, allora il risultato sarà un vettore che è ancora nella stessa direzione di $v$, solo che è allungato, accorciato o cambiato di verso a seconda del suo autovalore $\lambda$.

Quindi significa, che lungo certe direzioni (quelle degli autovettori) la matrice $S$ agisce semplicemente scalando sul vettore, senza cambiare la sua direzione.

Ipotizzando che la matrice $S$ abbia $n$ autovettori linearmente indipendenti, allora questi formano una base di $\mathbb{R}^n$ e quindi qualsiasi altro vettore in $\mathbb{R}^n$ può essere scritto come combinazione lineare di questi autovettori. Questo significa che se conosco come $S$ agisce sugli autovettori, allora so come agisce su qualsiasi vettore:
$$
\begin{aligned}
x &= 2v_1 + 4v_2 + 6v_3 \\[4pt]
Sx &= S(2v_1 + 4v_2 + 6v_3) \\[4pt]
&= 2Sv_1 + 4Sv_2 + 6Sv_3 \\[4pt]
&= 2\lambda_1 v_1 + 4\lambda_2 v_2 + 6\lambda_3 v_3 .
\end{aligned}
$$
Quindi gli autovalori mi dicono **quanto pesa ogni direzione**: se un autovalore è grande --> direzione importante, se l'autovalore è piccolo --> la direzione contribuisce pochissimo al risultato.

Intuitivamente stiamo dicendo che alcune direzioni spiegano molto, altre invece poco. Quelle poco importanti, come vedremo nell'applicazione di LSI, possono essere eliminate senza perdere troppa informazione.

La matrice $S$ è **simmetrica** quando $S = S^T$, ovvero quando è uguale alla sua trasposta. La simmetria è importante perché per questo tipo di matrici valgono alcune proprietà comode:
1. gli autovalori sono reali
2. gli autovettori associati ad autovalori diversi sono ortonormali (formano un angolo di 90° tra loro, in particolare il loro prodotto scalare è zero; normali significa che hanno lunghezza 1)
3. possono essere decomposte in modo pulito

Se inoltre $S$ è **semidefinita positiva** (ovvero $\forall w \in \mathbb{R}^n, w^T S w \geq 0$), allora **tutti i suoi autovalori sono non negativi** $\lambda_i \geq 0$. 

#### Eigen/diagonal decomposition
Sia $S \in \mathbb{R}^{n \times n}$ matrice quadrata con $n$ autovettori linearmente indipendenti. Allora è possibile decomporre $S$ come:
$$S = U \Lambda U^{-1}$$
dove le colonne di $U$ sono gli autovettori di $S$ e $\Lambda$ è una matrice diagonale con gli autovalori di $S$ sulla diagonale.

**Dimostrazione**: si prende semplicemente $U$ come matrice con colonne gli autovettori di $S$ e si moltiplica $SU$. Il risultato deriva direttamente dalla definizione di autovettore e dalla moltiplicazione per $U^{-1}$ da entrambe le parti:

<img src="img/eigen.png" alt="eigen decomposition" width="300"/>

Se $S$ è simmetrica, allora si arriva alla decomposizione ancora più pulita accennata prima: $S = Q \Lambda Q^T$. Questo perché $Q$ è matrice di autovettori ortonormali, quindi $Q^{-1} = Q^T$.

Esercizio:

<img src="img/eigen_ex.png" alt="eigen decomposition example" width="300"/>

Per capire se si può applicare la symmetric eigen decomposition, bisogna verificare che la matrice sia simmetrica. Questo ci porta a escludere automaticamente la prima e la terza matrice.

Riguardo la seconda matrice, troviamo anzitutto gli autovalori:
$$det(A - \lambda I) = det\begin{bmatrix} -\lambda & 1 \\ 1 & - \lambda \end{bmatrix} = \lambda^2 - 1 = 0 \Rightarrow \lambda_1 = 1, \lambda_2 = -1$$
dopodiché li normalizziamo (dividiamo per la loro lunghezza = $\sum_i v_i^2$) per ottenere gli autovettori ortonormali:
$$q_1 = \frac{1}{\sqrt{2}} \begin{bmatrix} 1 \\ 1 \end{bmatrix}, \qquad q_2 = \frac{1}{\sqrt{2}} \begin{bmatrix} 1 \\ -1 \end{bmatrix}$$
Quindi la matrice $Q$ è:
$$Q = \begin{bmatrix} \frac{1}{\sqrt{2}} & \frac{1}{\sqrt{2}} \\ \frac{1}{\sqrt{2}} & -\frac{1}{\sqrt{2}} \end{bmatrix}$$
e la matrice $\Lambda$ è:
$$\Lambda = \begin{bmatrix} 1 & 0 \\ 0 & -1 \end{bmatrix}$$
da cui $A = Q \Lambda Q^T$. La quarta matrice è più lunga falla sul quaderno, il procedimento è uguale.


#### **Singular Value Decomposition (SVD)**
Finora abbiamo parlato di decomposizione agli autovalori, tuttavia queste decomposizioni sono possibili **solo per matrici quadrate**. 

In IR con VSM invece, la matrice termini-documenti non è necessariamente quadrata ma spesso rettangolare: $A \in \mathbb{R}^{m \times n}$, dove $m$ è il numero di termini e $n$ è il numero di documenti. Per questo motivo per decomporre questa matrice usiamo una tecnica più generale, chiamata **Singular Value Decomposition (SVD)**, applicabile a qualsiasi matrice.

**SVD** dice che una matrice rettangolare $A$ può essere decomposta come prodotto di tre matrici:
$$A = U \Sigma V^T$$
dove :
- $U \in \mathbb{R}^{m \times m}$ è una matrice contenente come colonne gli autovettori di $AA^T$. Questa matrice è simmetrica $((AA^T)^T = AA^T)$ --> gli autovettori sono ortonormali
- $V \in \mathbb{R}^{n \times n}$ è una matrice contenente come colonne gli autovettori di $A^TA$. Anche questa matrice è simmetrica --> autovettori ortonormali
- $\Sigma \in \mathbb{R}^{m \times n}$ è una matrice che contiene i valori singolari di $A$, ossia $\sigma_i = \sqrt{\lambda_i}$ (dove $\lambda_i$ è l'i-esimo autovalore di $AA^T$ o $A^TA$) lungo la diagonale

**Intuitivamente, cosa rappresentano $U$ e $V$**? Poiché A è una matrice termini-documenti:
 - $U = AA^T$ **è una matrice che rappresenta la relazione termine-termine**.   
 Questa matrice misura, in un certo senso, quanto i termini sono collegati tra loro attraverso i documenti: due termini saranno collegati se compaiono in documenti simili.  
 es. "laptop" e "computer" avranno probabilmente una relazione alta perché compaiono spesso negli stessi documenti o in contesti simili, lo capiamo guardando il seguente esempio semplificato contando solo la presenza dei termini nei documenti:

    <img src="img/term-term.png" alt="term-term matrix" width="300"/>
    
    Nella matrice $A$, la riga "laptop" è $(1,1,0)$ (laptop compare nei primi due documenti), mentre la riga "computer" è $(1,1,1)$. Calcolando $AA^T$, otteniamo che il valore nella cella $(AA^T)_{laptop, computer}$ è $1\cdot1 + 1\cdot1 + 0\cdot1 = 2$, ecco perché $AA^T$ misura una relazione tra i termini.

    Intuitivamente $AA^T$ quindi contiene tutti autovettori che rappresentano le direzioni importanti nello spazio dei termini, più precisamente **direzioni latenti nello spazio dei termini**, ossia direzioni che rappresentano un pattern di associazione tra termini diversi. 
- $V = A^TA$ **è una matrice che rappresenta la relazione documento-documento**.  
    Il discorso è analogo a quello fatto per $U$, ma questa volta misurando quanto i documenti sono simili tra loro in base ai termini che contengono (due documenti saranno collegati se contengono termini simili o pattarn simili di termini).

**Cos'è $\Sigma$**? I valori singolari di $A$ sono le radici quadrate degli autovalori di $AA^T$ (o equivalentemente di $A^TA$). Questi valori singolari rappresentano l'importanza di ogni direzione latente: più è grande un valore singolare, più la direzione latente associata è importante per rappresentare le relazioni nei dati. 

I valori singolari sono ordinati in modo non crescente sulla diagonale di $\Sigma$: $\sigma_1 \geq \sigma_2 \geq ... \geq \sigma_r > 0$, dove $r$ è il rango di $A$ (il rango indica il numero di dimensioni indipendenti della matrice, e quindi il numero di valori singolari non nulli). L'ordinamento è importante **affinché sappiamo che le dimensioni più importanti sono quelle associate ai primi valori singolari**, di modo come vedremo di poter ridurre la dimensionalità mantenendo solo i primi $k$ valori singolari più grandi.

Nel contesto termini-documenti il concetto di rango è importante perché indica quante direzioni latenti indipendenti (ossia non ridondanti, non derivano dalla combinazione lineare di altre) esistono effettivamente nei dati. Quindi, anche se si parte da uno spazio originale enorme, l'informazione che ci interessa potrebbe essere rappresentata efficacemente in uno spazio più piccolo.

Importante ribadire che quindi con la **SVD non abbiamo cambiato la matrice $A$, ma l'abbiamo semplicemente riscritta come prodotto di tre matrici**.

<img src="img/svd.png" alt="SVD decomposition" width="400"/>

#### **Low-Rank Approximation**
Abbiamo visto che quindi $A$ può essere scomposta in un prodotto di tre matrici più semplici e significative tramite SVD. Ci chiediamo ora se **sia possibile sostituire la matrice originale $A$ con una più semplice $A_k$, di rango $k$, che sia quanto più vicina possibile ad $A$**.

**Perché questo**? Perché in IR la matrice originale $A$ è spesso troppo grande, sparsa e rumorosa. **Vogliamo ottenere una nuova matrice $A_k$ che conservi le strutture importanti ma elimini il rumore.**

Formalmente il problema che ci stiamo ponendo quindi è il seguente:
$$A_k = argmin_{rank(X)=k} ||A - X||_F$$
**ossia stiamo cercando, tra tutte le matrici $X$ di rango $k$, quella più vicina ad $A$**. La distanza viene misurata in termini di norma di Frobenius:
$$||A||_F = \sqrt{\sum_{i=1}^m \sum_{j=1}^n a_{ij}^2}$$
(ossia considero come distanza semplicemente la differenza tra la somma sotto radice dei quadrati degli elementi di $A$ e di $X$).

La strategia per ottenere $A_k$ che soddisfi la formula di prima è la **Low-Rank Approximation** che abbiamo citato prima: semplicemente, sapendo di aver ordinato in ordine non crescente i valori singolari di $A$ nella matrice $\Sigma$, allora per ottenere $A_k$ basta **mantenere solo i primi $k$ valori singolari più grandi, e azzerare tutti gli altri**:
$$A_k = U diag(\sigma_1, \sigma_2, ..., \sigma_k, 0, ..., 0) V^T$$
In pratica quindi ottengo $A_k$ con la seguente formula:
$$A_k = \sum_{i=1}^k \sigma_i u_i v_i^T = \sigma_1 u_1 v_1^T + \sigma_2 u_2 v_2^T + ... + \sigma_k u_k v_k^T$$
dove $u_i$ è l'i-esimo autovettore di $AA^T$ e $v_i$ è l'i-esimo autovettore di $A^TA$ (ossia le i-esime colonne di $U$ e $V$).  
In questa formula l'i-esimo pezzo $\sigma_i u_i v_i^T$ è una matrice di rango 1 (prodotto di un vettore colonna per un vettore riga) che rappresenta una direzione latente nello spazio dei termini e dei documenti, e il peso di questa direzione è dato dal valore singolare $\sigma_i$. Sono i primi pezzi quelli più importanti perché sono quelli con valori singolari più grandi.

Si può dimostrare che questa approssimazione è **ottimale** in norma di Frobenius, nel senso che **non esiste nessun'altra matrice di rango $k$ che sia più vicina ad $A$ di $A_k$**. 

Chiaramente, l'errore di approssimazione diminuisce al crescere di $k$: più valori singolari mantengo, più $A_k$ si avvicina ad $A$. Tuttavia, mantenere più valori singolari significa anche mantenere più rumore, quindi c'è un trade-off da considerare.

### **Latent Semantic Indexing (LSI)**
Ci siamo quindi calcolati $A_k$, approssimazione di $A$ di rango $k$. Ora, come usiamo questa matrice per fare IR?

In $A_k$ esistono ancora righe associate ai termini e colonne associate ai documenti, ma a questo punto **i documenti vivono in uno spazio latente di $k \ll r$ dimensioni**. Se prima avevamo ad esempio 50.000 termini e 10.000 documenti --> ogni documento era un vettore a 50.000 dimensioni. Con LSI si scende a uno spazio di $k$ dimensioni, come ad esempio 100, 200 o 300.

Ognuna delle dimensioni di questo spazio **non sono gli assi originali**: mentre nel VSM classico ogni asse è un termine, in LSI ognuna delle $k$ dimensioni è detta **dimensione latente** e rappresenta un pattern di associazione tra termini.

Ad esempio, una dimensione latente potrebbe avere pesi alti per "laptop", "software", "display", "computer", e noi potremmo interpretarla come una dimensione legata al tema "informatica".  
Sia però chiaro il fatto che LSI non associa davvero etichette umane alle dimensioni, semplicemente trova direzioni matematiche in base alla matrice termine-documento $A$ che riassumono i pattern forti di associazione tra termini e documenti.

Introdurre LSI ha senso perché VSM classico ha sì molti vantaggi, come:
- Partial matching (posso trovare documenti che non contengono esattamente i termini della query ma termini simili)
- Estensioni con relevance feedback e Rocchio
- Fondamento geometrico chiaro
- ranking basato su similarità

Però VSM ha anche **due grossi problemi**:
1. **Polisemia**: se una parola ha più significati.  
Prendiamo come esempio la parola "Saturn" (pianeta ma anche azienda automobilistica). Dato che nel VSM classico ogni parola rappresenta una dimensione, un modello può considerare simili anche documenti che parlano di argomenti completamente diversi, semplicemente perché contengono la parola "Saturn".
2. **Sinonimia**: se due parole diverse hanno lo stesso significato.  
Prendiamo come esempio "car" e "automobile". Se un documento parla di "car" e un altro di "automobile" e non hanno molte parole in comune, allora il VSM classico li considererà come documenti poco simili, anche se in realtà parlano dello stesso argomento.

**LSI cerca di risolvere i problemi costruendo uno spazio in cui termini simili e documenti simili finiscono tutti vicini, seguendo la stessa dimensione latente**. In questo modo termini e documenti semanticamente collegati, anche senza condividere esattamente le stesse parole, risulteranno collegati in quanto vicini nello spazio latente.

Come si vede nell'immagine sotto quindi il nuovo spazio latente permette la rappresentazione non solo dei documenti nello spazio, ma anche dei termini.

<img src="img/lsi.png" alt="LSI" width="400"/>

**Intuizione con i blocchi**: se una matrice $A \in \mathbb{R}^{m \times n}$ ha ad esempio un gruppo di termini e documenti che riguarda il topic 1, un altro gruppo che riguarda il topic 2, e così via fino al topic $k$, allora la matrice ha una struttura a blocchi dove i termini che parlano di un certo topic avranno peso solo per i documenti che lo riguardano e 0 per gli altri documenti etc..  
Nell'esempio si vede infatti come il blocco 1 contiene termini riguardanti automobili etc... fuori dal blocco 1 parole come "car" e "automobile" avranno poche non-zero entries

<img src="img/blocks.png" alt="blocks" width="300"/>

Proprio per questo ha senso "far collassare" (approssimare) la matrice in uno spazio latente di $k$ dimensioni, dove ogni dimensione rappresenta un topic latente. Chiaramente la scelta di $k$ è importante in questo senso e deve essere soggetta a tuning su benchmark.

#### **LSI in pratica**
In pratica quindi LSI funziona come segue:
1. Si parte dalla matrice termini-documenti $A \in \mathbb{R}^{m \times n}$, e si applica la SVD per decomporla in $A = U \Sigma V^T$.
2. Si applica la low-rank approximation per ottenere $A_k = U_k \Sigma_k V_k^T$, dove $U_k$ contiene solo i primi $k$ autovettori di $AA^T$, $\Sigma_k$ contiene solo i primi $k$ valori singolari, e $V_k$ contiene solo i primi $k$ autovettori di $A^TA$.

A questo punto, i vari pezzi hanno le seguenti dimensioni, riportando correttamente $A_k$ a dimensione $m \times n$:
- $U_k \in \mathbb{R}^{m \times k}$
- $\Sigma_k \in \mathbb{R}^{k \times k}$
- $V_k^T \in \mathbb{R}^{k \times n}$

Quindi $A_k$ non rappresenta propriamente lo spazio latente, ma la matrice termini-documenti approssimata utilizzandolo (infatti lo spazio latente ha dimensione $k$, mentre $A_k$ ha ancora dimensione $m \times n$).  
**Le coordinate vere e proprie dello spazio latente, che poi utilizziamo in pratica per calcolare la similarità tra documenti e query, si trovano nei fattori della SVD**: $U_k$, $\Sigma_k$ e $V_k$.

In particolare:
- i documenti nello spazio latente sono tipicamente rappresentati come 
    $$D_k = \Sigma_k V_k^T$$
    vedendo le dimensioni infatti, torna proprio che $D_k \in \mathbb{R}^{k \times n}$, quindi ogni colonna è un documentorappresentato nello spazio latente di $k$ dimensioni. Il punto di moltiplicare per $\Sigma_k$ è sempre lo stesso: scalare le dimensioni latenti in base alla loro importanza (valori singolari più grandi --> dimensioni più importanti).
- i termini nello spazio latente sono tipicamente rappresentati come
    $$T_k = U_k \Sigma_k$$
    vedendo le dimensioni infatti, torna proprio che $T_k \in \mathbb{R}^{m \times k}$, quindi ogni riga è un termine rappresentato nello spazio latente di $k$ dimensioni.

Quindi ora è chiaro che sia termini che documenti sono rappresentati nello stesso spazio a $k$ dimensioni. Ci chiediamo ora **come calcolare la similarità tra due documenti $d_i$ e $d_j$ nello spazio latente**.

Si torna alle classiche misure di similarità, uno dei modi è utilizzare la **cosine similarity** ($\frac{d_i^T d_j}{\|d_i\| \|d_j\|}$).

Altra domanda fondamentale: **come proiettare una query nello spazio latente, per poi poterla confrontare con i documenti?**  
Sappiamo che se un vocabolario ha $m$ termini, allora $q \in R^m$ è la rappresentazione della query nello spazio originale. Per portarla nello spazio LSI si usa una delle due seguenti formule, che sono equivalenti:
$$q_k^T = q^T U_k \Sigma_k^{-1} \qquad q_k = \Sigma_k^{-1} U_k^T q$$
(si arriva quindi con la formula a $q_k \in \mathbb{R}^k$, come ci aspettiamo).  
Una volta proiettata la query, per trovare il ranking è di base necessario calcolare la similarità tra $q_k$ con tutti i documenti in $D_k$, esistono però metodi di ottimizzazione per evitarlo.

Importante notare come il passaggio della query nello spazio latente fa sì che questa passi dall'essere un vettore sparso a un vettore molto denso, dal momento che $k \ll m$ e che dopo la trasformazione può avere più valori distribuiti sulle varie dimensioni latenti.  
Quindi una query come "laptop" dopo la trasformazione può attivare dimensioni collegate a "computer", "software", "display" etc... una volta mappata nello spazio latente

**Evidenza empirica**: negli anni '90, quando LSI è stato sviluppato, era molto costoso da calcolare. Tipicamente $k \le 200$, quindi 200 dimensioni latenti, erano considerate insoddisfacenti. Si ottenevano risultati validi tra le 250 e le 350 dimensioni. 

In generale con LSI ci si aspetta un **miglioramento della recall** (perché riesce a recuperare documenti che non contengono esattamente i termini della query ma semanticamente termini simili), con Precision sopra la mediana TREC (dove TREC standar di riferimento per IR). Si vede nella tabella delle slide che aumentando le dimensioni da 250 a 346, la precision passa da 0.367 a 0.374. 

#### **Limiti di LSI**
Tra i principali problemi legati ad LSI:
1. **Negazioni**: LSI non è in grado di gestire frasi negate, del tipo "documenti su jaguar ma non la macchina". Dal momento che infatti LSI lavora con similarità vettoriale e associazioni semantiche, non è bravo a gestire vincoli logici come "non". Se un termine è presente in una query, ma con un "non" prima, LSI lo tratterà comunque come semanticamente rilevante
2. **Query Booleane**: chiaramente LSI non è pensato per risolvere query booleane rigide: se gli chiedo di restituire documenti che contengono esattamente certe parole mi restituirà anche documenti correlati semanticamente ma che non le contengono, per tutto il meccanismo visto prima.
3. **Può peggiorare la precision percepita**: se da un lato LSI migliora la recall, dall'altro può peggiorare la precision, restituendo documenti che sono semanticamente correlati ma non rilevanti per la query.  
es. il prof aveva fatto un motore di ricerca basato su LSI per un'azienda petrolifera. Cercando "pompa" il primo risultato è stato "albero di Natale", perché in gergo tecnico l'albero di Natale è un tipo di pompa usata in ambito petrolifero. Tuttavia questo risultato, per un utente normale, è di difficile interpretazione e quindi peggiora la precision percepita.

**Legame con clustering**: nell'esempio dei blocchi di prima abbiamo visto come ogni dimensione dello spazio latente rappresenta un pattern tra termini e documenti, può quindi intuitivamente rappresentare un topic. Questo è vicino all'idea del clustering: raggruppare insieme documenti/termini legati ad uno stesso argomento. 

**N.B.** come gestisce LSI termini polisemici? Intuitivamente il termine sarà proiettato nello spazio latente puntando parzialmente verso tutte le dimensioni latenti che rappresentano i vari significati del termine. Ad esempio, "Saturn" sarà proiettato verso la dimensione latente legata al pianeta e verso la dimensione latente legata all'azienda automobilistica. Attenzione al fatto che LSI non crea quindi due rappresentazioni separate dello stesso termine, il vettore è solo uno!

**Altre applicazioni di LSI**: LSI e SVD non sono una tecnica utile solo per testi, ma in generale funzionano per qualsiasi situazione in cui si ha a che fare con una matrice **features x objects** (nel caso di IR le feauters sarebbero i termini e gli objects i documenti: ogni documento è descritto dalle parole che contiene).  
es. matrice A = film x utenti, dove ogni cella rappresenta quanto l'utente ha apprezzato quel film. Potrebbe avere senso anche qui applicare SVD in quanto questa matrice potrebbe avere dimensionalità ridondante, con dimensioni latenti che potrebbero essere qualcosa del tipo "film d'azione", "film romantici" etc... e quindi proiettare utenti e film in questo spazio latente per fare raccomandazioni più intelligenti.

Inoltre LSI può anche essere usato per classificazione: se un utente non ha ancora votato un film, se so che si trova vicino ad altri utenti che gli hanno dato un certo voto o che ha dato buoni voti ad altri film di quel genere , potrei provare a stimare il suo voto mancante.

**N.B.** Abbiamo visto come $A_k$ sia una matrice che diventa densa per via dell'approssimazione --> non posso più sfruttare una rappresentazione sparsa per memorizzarla, diventa intrattabile quando parliamo di grandi collezioni. Infatti per LSI in realtà non ci interessa memorizzare $A_k$, ma conoscere solo i documenti e i termini nello spazio latente, quindi **mi basta fare la decomposizione e l'approssimazione senza dover materializzare $A_k$**.  
Inoltre LSI è molto interessante perché **esistono diversi algoritmi iterativi che permettono di calcolare solo i primi $k$ autovettori e valori singolari (partendo dal più grande), senza dover calcolare tutta la SVD --> ancora più efficiente**